# I. Data Model Selection
The core objective is to build a prediction model based on the finalized modeling dataset (model_df) to classify healthcare needs of respondents aged 60 and above. This means classifying respondents into the **lower healthcare needs group (target=0)** or **higher healthcare needs group (target=1)**, providing model support for subsequent analysis and applications.

## 1. Logistic Regression = Baseline Model
- **Task**: Binary classification; provides a simple, interpretable risk score for predicting whether an older adult belongs to the higher/lower healthcare needs group.
- **Advantages**:
  - Stable, simple, and fast to train.
  - High interpretability; directly shows the direction and magnitude of each feature’s impact on “high healthcare needs.”

## 2. Random Forest = Classification via Multiple Decision Tree Voting
- **Task**:
  - Binary classification; input: 15 features in model_df (age, gender, diseases, smoking, cardiovascular conditions, etc.); output: 0 or 1 classification result.
- **Advantages**:
  - Designed for binary classification, naturally suitable for predicting high/low healthcare needs.
  - Captures non-linear relationships and multi-disease comorbidity effects (ideal for medical data).
  - Outputs feature importance to identify key factors driving high healthcare needs.
  - Less prone to overfitting and more reliable than single decision trees.
  - Requires minimal data preprocessing (normalization/standardization optional).
  - Robust to missing values and outliers.
  - More interpretable than XGBoost and more accurate than logistic regression.

## 3. XGBoost = Final High-Precision Scoring Model
- **Task**: Fit non-linear relationships and complex interactions to accurately predict older adults with high healthcare needs.
- **Advantages**:
  - Highest accuracy among the three models (gradient boosting, strongest predictive power).
  - Suitable for imbalanced, medical, and health risk prediction data.
  - Noise-resistant and strong generalization ability.

---

# II. Training & Test Set Split Instructions
## 1. Split Basis
- Use the cleaned final modeling dataset model_df, which has completed variable selection, feature engineering, and missing value handling and is ready for model training.
- Feature variables: All predictive variables in model_df except target.
- Stratified sampling: Split by target to ensure the ratio of high/low healthcare needs groups in training and test sets matches the original data, avoiding evaluation bias.
- Reproducibility: Fixed random seed to ensure consistent split results across runs.

## 2. Split Principles
- Target variable: target (0 = low healthcare needs, 1 = high healthcare needs).
- Feature variables: All predictive variables in model_df except target.
- Stratified sampling: Split by target to preserve class distribution.
- Reproducibility: Fixed random seed.

## 3. Split Ratio
- Training set (80%): For training Logistic Regression, Random Forest, and XGBoost models.
- Test set (20%): For testing model performance and evaluating prediction results.

---

# III. Model Training
## Model 1: Logistic Regression
### a) Training Procedure
- Use model_df.
- Features X: All predictive variables (exclude target and HUQ051).
- Target y: target.
- Train on training set, evaluate on test set.
- Output metrics: accuracy, AUC, etc.

### b) Code
```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# Predict
y_pred_lr = lr.predict(X_test)

# Evaluate
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
```

## Model 2: Random Forest
### a) Training Procedure
- Use final modeling data: model_df.
- Define X and y.
- X: All predictive variables (15 features).
- y: Target variable target (0/1).

### b) Code
```python
# 1. Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 2. Load modeling data
# model_df = pd.read_csv("your_data.csv")

# 3. Define features X and target y
X = model_df.drop(columns=["target"])
y = model_df["target"]

# 4. Split train/test set (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Preserve class ratio
)

# 5. Build Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=None,
    min_samples_split=2
)

# 6. Train model
rf_model.fit(X_train, y_train)

# 7. Predict on test set
y_pred = rf_model.predict(X_test)

# ===================== Model Evaluation =====================
print("===== Random Forest Binary Classification Evaluation =====")
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ===================== Feature Importance =====================
print("\n===== Feature Importance Ranking =====")
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)
print(feature_importance)
```

## Model 3: XGBoost
### a) Training Procedure
- Final prediction model.
- Use the same train/test split.
- Output highest-accuracy classification results.

### b) Code
```python
from xgboost import XGBClassifier

# Train
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='log_loss')
xgb.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb.predict(X_test)

# Evaluate
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))
```


